# Ćwiczenie 4.4: Gradient Boosting i XGBoost — drzewa, które poprawiają poprzednie drzewa

W poprzednim notebooku Random Forest budował **wiele drzew niezależnie** i łączył ich głosy.

Tutaj robimy inny krok: budujemy drzewa **po kolei**. Każde następne drzewo próbuje poprawić błędy aktualnego modelu.

```text
Random Forest:
    wiele drzew równolegle / niezależnie -> głosowanie lub średnia

Gradient Boosting:
    model startowy -> błędy -> drzewo-poprawka -> aktualizacja -> nowe błędy -> ...
```

Pracujemy na problemie churn: czy klient odejdzie z usługi?

Plik: **wersja dla studentów**.



## Jak korzystać z tego notebooka?

Ten notebook domyka przejście od pojedynczych drzew i Random Forest do boostingu. Najważniejsze pytanie brzmi:

```text
co dokładnie robią kolejne drzewa, skoro one nie głosują jak w Random Forest?
```

Czytaj notebook w tej kolejności:

1. **Sekcja 1** — rozbij algorytm boostingu na części: model startowy, residuale, małe drzewo, `learning_rate`, aktualizacja.
2. **Sekcja 2** — wykonaj jedną rundę boostingu półręcznie.
3. **Sekcja 2B** — zobacz trzy kolejne stump-y i wartości zapisane w ich liściach.
4. **Sekcja 2C** — sprawdź, jak gotowy model przewiduje nowego klienta. Tu jest kluczowy wniosek: **nie wybieramy jednego drzewa; sumujemy wszystkie drzewa**.
5. **Sekcja 2D** — porównaj duży i mały `learning_rate`, a potem zobacz, co daje więcej iteracji.
6. **Sekcja 3** — zobacz, co XGBoost dodaje do zwykłego Gradient Boostingu.
7. **Sekcje praktyczne** — porównaj modele, progi decyzyjne, ważność cech i ranking ryzyka.

Najważniejsza intuicja na cały notebook:

```text
Random Forest = wiele drzew niezależnie + głosowanie/uśrednianie
Gradient Boosting = wiele drzew po kolei + suma poprawek
XGBoost = boosting drzew + bardziej formalne liczenie poprawek i regularizacja
```


## 0. Mapa: co zmienia się względem poprzednich notebooków?

Najważniejsze: **boosting nie zmienia sposobu działania pojedynczego splitu w drzewie**.

Pojedyncze drzewo nadal może pytać:

```text
liczba_reklamacji <= 1.5?
umowa_miesieczna = 1?
srednie_logowania_tyg <= 3.2?
```

Nowość jest gdzie indziej: w tym, **jak wiele małych drzew jest składanych w jeden model**.

```text
Notebook 4.1
────────────
cechy binarne
      ↓
Gini
      ↓
jedno drzewo klasyfikacyjne
      ↓
klasa

Notebook 4.2A
─────────────
cechy liczbowe
      ↓
progowanie
      ↓
Gini
      ↓
jedno drzewo klasyfikacyjne
      ↓
klasa

Notebook 4.3
────────────
wiele niezależnych drzew
      ↓
bootstrap + losowanie cech
      ↓
głosowanie / średnia prawdopodobieństw
      ↓
klasa lub prawdopodobieństwo

Notebook 4.4
────────────
wiele małych drzew budowanych sekwencyjnie
      ↓
każde drzewo poprawia poprzedni model
      ↓
learning_rate kontroluje wielkość poprawki
      ↓
prawdopodobieństwo churn
```

W skrócie:

| Model | Co jest główną ideą? |
|---|---|
| Decision Tree | jedno drzewo decyzyjne |
| Random Forest | wiele niezależnych drzew i głosowanie |
| Gradient Boosting | wiele małych drzew dodawanych sekwencyjnie |
| XGBoost | Gradient Boosting z mocniejszą kontrolą złożoności, gradientami/Hessianami i regularizacją |


Dodatkowe rozróżnienie, które często porządkuje temat:

| Pytanie | Random Forest | Gradient Boosting / XGBoost |
|---|---|---|
| Czy drzewa znają wyniki poprzednich drzew? | Nie | Tak |
| Czy kolejność drzew ma znaczenie? | Nie | Tak |
| Czy pojedyncze drzewo ma być mocne? | Często może być głębsze | Zwykle jest małe/płytkie |
| Co robi `learning_rate`? | Zwykle nie występuje | Zmniejsza wkład każdego kolejnego drzewa |
| Co oznacza dodanie kolejnych drzew? | więcej głosów | więcej sekwencyjnych poprawek |


## 1. Algorytm boostingu w częściach

W klasyfikacji binarnej model boostingowy zwykle nie dodaje bezpośrednio prawdopodobieństw. Dodaje tzw. **score** albo **logit**, a potem zamienia go na prawdopodobieństwo funkcją sigmoid.

```text
score F(x)  ->  sigmoid(F(x))  ->  prawdopodobieństwo klasy "tak"
```

Schemat jednego przebiegu:

```text
1. Zacznij od prostej predykcji F0.
   Na przykład: wszyscy klienci mają takie samo prawdopodobieństwo odejścia.

2. Zamień score na prawdopodobieństwo:
       p = sigmoid(F)

3. Policz, gdzie model się myli.
   Dydaktycznie: residual = y - p.
   Jeżeli y=1 i p jest małe, residual jest dodatni -> trzeba podnieść ryzyko.
   Jeżeli y=0 i p jest duże, residual jest ujemny -> trzeba obniżyć ryzyko.

4. Zbuduj małe drzewo, które przewiduje poprawkę.

5. Dodaj tylko mały kawałek tej poprawki:
       F_new = F_old + learning_rate * poprawka_z_drzewa

6. Powtarzaj wiele razy.
```

To jest dobry moment, żeby rozbić algorytm na partie. Student powinien osobno rozumieć:

1. predykcję startową,
2. residuale / gradienty,
3. małe drzewo-poprawkę,
4. aktualizację przez `learning_rate`,
5. sumowanie wielu drzew,
6. różnicę między Gradient Boosting i XGBoost.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    classification_report,
    ConfusionMatrixDisplay,
)

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 60)


def make_one_hot_encoder():
    """Kompatybilność ze starszymi i nowszymi wersjami scikit-learn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def logit(p):
    p = np.asarray(p)
    return np.log(p / (1 - p))


def is_missing_todo(value):
    return value is Ellipsis


def all_filled(*values):
    return not any(is_missing_todo(value) for value in values)


def check_close(name, student_value, reference_value, atol=1e-6):
    """Mały pomocnik do sprawdzania ręcznych obliczeń."""
    if is_missing_todo(student_value):
        print(f"{name}: uzupełnij wartość w miejscu wielokropka (...).")
    elif np.isclose(student_value, reference_value, atol=atol):
        print(f"{name}: OK")
    else:
        print(f"{name}: sprawdź wynik. Wpisano {student_value}, a kod kontrolny ma {reference_value:.6f}.")


try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as exc:
    XGB_AVAILABLE = False
    XGB_IMPORT_ERROR = exc

print("XGBoost dostępny:", XGB_AVAILABLE)


## 2. Mini-przykład: jedna runda boostingu półręcznie

Zaczynamy od bardzo małych danych. Chodzi o mechanikę, nie o wynik produkcyjny.

Mamy 8 klientów. `odejdzie_num = 1` oznacza churn, a `0` oznacza brak churn.

Na tym przykładzie zrobimy jedną rundę:

```text
predykcja startowa
      ↓
residuale y - p
      ↓
jedno małe drzewo-poprawka
      ↓
aktualizacja prawdopodobieństw
```


### Uwaga dydaktyczna: ten mini-przykład jest uproszczeniem

W tej części używamy intuicyjnej poprawki:

```text
residual = y - p
poprawka w liściu = średni residual w liściu
```

To bardzo dobrze pokazuje ideę boostingu: **kolejne drzewo poprawia błędy aktualnego modelu**.

Pełne implementacje, takie jak `GradientBoostingClassifier` i szczególnie XGBoost, liczą poprawki bardziej formalnie przez pochodne funkcji straty. Dlatego nie należy traktować tego mini-przykładu jako dokładnej reimplementacji bibliotek, tylko jako ręczny model mentalny algorytmu.

In [ ]:
mini = pd.DataFrame({
    "klient": ["A", "B", "C", "D", "E", "F", "G", "H"],
    "miesiace_umowy": [60, 45, 30, 20, 10, 8, 6, 4],
    "liczba_reklamacji": [0, 0, 1, 1, 2, 3, 2, 4],
    "srednie_logowania_tyg": [6.5, 5.8, 4.2, 3.0, 2.5, 1.5, 2.0, 1.0],
    "odejdzie_num": [0, 0, 0, 0, 1, 1, 1, 0],
})

display(mini)

# W wersji studenckiej nie drukujemy tutaj średniego churnu,
# bo to jest pierwszy wynik do policzenia w ćwiczeniu A1.


In [ ]:
# Ćwiczenie A1: predykcja startowa
#
# W klasyfikacji binarnej Gradient Boosting zaczyna od stałej predykcji.
# Najprostsza intuicja:
#     p0 = odsetek klasy 1 w zbiorze treningowym
#
# Potem zamieniamy p0 na score/logit:
#     F0 = log(p0 / (1 - p0))
#
# TODO: policz ręcznie na podstawie tabeli mini.
# 1. Ile jest obserwacji z odejdzie_num == 1?
# 2. Ile jest wszystkich obserwacji?
# 3. base_rate = liczba_jedynek / liczba_obserwacji
# 4. score_start = log(base_rate / (1 - base_rate))
# 5. p0 = sigmoid(score_start), czyli kontrolnie powinno wrócić base_rate.

base_rate_recznie = ...
score_start_recznie = ...
p0_recznie = ...

base_rate_ref = mini["odejdzie_num"].mean()
score_start_ref = float(logit(base_rate_ref))
p0_ref = float(sigmoid(score_start_ref))

check_close("base_rate", base_rate_recznie, base_rate_ref)
check_close("score_start", score_start_recznie, score_start_ref)
check_close("p0", p0_recznie, p0_ref)


### Ćwiczenie A2: residuale, czyli co model ma poprawić

Dla uproszczonej intuicji liczymy:

```text
residual = y - p
```

Interpretacja:

```text
y = 1, p za małe  -> residual dodatni -> model powinien podnieść score
y = 0, p za duże  -> residual ujemny  -> model powinien obniżyć score
```

W prawdziwym Gradient Boostingu mówi się o **ujemnym gradiencie funkcji straty**. Dla log-loss w klasyfikacji binarnej ta intuicja jest bardzo bliska: poprawka wskazuje kierunek, w którym trzeba zmienić aktualny score.


In [ ]:
mini_step = mini.copy()
mini_step["p0"] = p0_ref
mini_step["residual_y_minus_p"] = mini_step["odejdzie_num"] - mini_step["p0"]

# TODO A2: policz dwie wartości ręcznie.
#
# Dla klienta z y=1:
#     residual = 1 - p0
#
# Dla klienta z y=0:
#     residual = 0 - p0

residual_gdy_y_1_recznie = ...
residual_gdy_y_0_recznie = ...

residual_y_1_ref = 1 - p0_ref
residual_y_0_ref = 0 - p0_ref

check_close("residual dla y=1", residual_gdy_y_1_recznie, residual_y_1_ref)
check_close("residual dla y=0", residual_gdy_y_0_recznie, residual_y_0_ref)

display(mini_step[["klient", "odejdzie_num", "p0", "residual_y_minus_p"]])


### Ćwiczenie A3: małe drzewo-poprawka

Wyobraźmy sobie, że pierwsze małe drzewo ma tylko jeden split:

```text
liczba_reklamacji >= 2 ?
```

To drzewo nie przewiduje jeszcze klasy `tak/nie`. Ono przewiduje **poprawkę do aktualnego score**.

W uproszczonej wersji policzymy wartość liścia jako średni residual w danej gałęzi:

```text
wartość_liścia = średnia(residual_y_minus_p w liściu)
```

To nie jest pełna implementacja scikit-learn ani XGBoost, ale bardzo dobrze pokazuje sens: **drzewo ma powiedzieć, w którą stronę przesunąć predykcję**.


In [ ]:
mask_duzo_reklamacji = mini_step["liczba_reklamacji"] >= 2

# TODO A3: policz średnie residuale w dwóch liściach.
#
# Liść 1: liczba_reklamacji < 2
#     weź residuale klientów A, B, C, D
#     policz ich średnią
#
# Liść 2: liczba_reklamacji >= 2
#     weź residuale klientów E, F, G, H
#     policz ich średnią

poprawka_malo_reklamacji_recznie = ...
poprawka_duzo_reklamacji_recznie = ...

poprawka_malo_reklamacji_ref = mini_step.loc[~mask_duzo_reklamacji, "residual_y_minus_p"].mean()
poprawka_duzo_reklamacji_ref = mini_step.loc[mask_duzo_reklamacji, "residual_y_minus_p"].mean()

check_close("poprawka: reklamacje < 2", poprawka_malo_reklamacji_recznie, poprawka_malo_reklamacji_ref)
check_close("poprawka: reklamacje >= 2", poprawka_duzo_reklamacji_recznie, poprawka_duzo_reklamacji_ref)

print("Interpretacja:")
print("- ujemna poprawka obniża ryzyko churn")
print("- dodatnia poprawka podnosi ryzyko churn")


### Ćwiczenie A4: aktualizacja predykcji przez `learning_rate`

Teraz dodajemy poprawkę do score:

```text
F1 = F0 + learning_rate * poprawka
p1 = sigmoid(F1)
```

`learning_rate` mówi, jak odważnie model ma przyjąć poprawkę.

- duży `learning_rate` → szybciej się dopasowujemy, ale łatwiej przesadzić,
- mały `learning_rate` → ostrożniejsze kroki, zwykle potrzeba więcej drzew.

W prawdziwych modelach często używa się np. `0.1`, `0.05`, `0.03`. Tutaj użyjemy większej wartości, żeby efekt był łatwo widoczny w małej tabeli.


In [ ]:
learning_rate_mini = 0.8

# TODO A4: policz ręcznie nowe score i prawdopodobieństwa dla dwóch liści.
#
# Dla liścia "mało reklamacji":
#     F1_malo = F0 + learning_rate * poprawka_malo_reklamacji
#     p1_malo = sigmoid(F1_malo)
#
# Dla liścia "dużo reklamacji":
#     F1_duzo = F0 + learning_rate * poprawka_duzo_reklamacji
#     p1_duzo = sigmoid(F1_duzo)

F1_malo_recznie = ...
p1_malo_recznie = ...
F1_duzo_recznie = ...
p1_duzo_recznie = ...

F1_malo_ref = score_start_ref + learning_rate_mini * poprawka_malo_reklamacji_ref
p1_malo_ref = float(sigmoid(F1_malo_ref))
F1_duzo_ref = score_start_ref + learning_rate_mini * poprawka_duzo_reklamacji_ref
p1_duzo_ref = float(sigmoid(F1_duzo_ref))

check_close("F1 mało reklamacji", F1_malo_recznie, F1_malo_ref)
check_close("p1 mało reklamacji", p1_malo_recznie, p1_malo_ref)
check_close("F1 dużo reklamacji", F1_duzo_recznie, F1_duzo_ref)
check_close("p1 dużo reklamacji", p1_duzo_recznie, p1_duzo_ref)

# Tabela kontrolna liczona kodem.
mini_update = mini_step.copy()
mini_update["poprawka_z_drzewa"] = np.where(
    mask_duzo_reklamacji,
    poprawka_duzo_reklamacji_ref,
    poprawka_malo_reklamacji_ref,
)
mini_update["F1"] = score_start_ref + learning_rate_mini * mini_update["poprawka_z_drzewa"]
mini_update["p1"] = sigmoid(mini_update["F1"])

display(mini_update[[
    "klient", "liczba_reklamacji", "odejdzie_num", "p0",
    "poprawka_z_drzewa", "p1"
]].round(3))


### Ćwiczenie A5: zakoduj jedną linijkę aktualizacji

Teraz to samo zapisujemy jako funkcję. To jest najważniejsza linijka całego boostingu:

```text
nowy_score = stary_score + learning_rate * poprawka
```


In [ ]:
def boosting_update_student(old_score, tree_correction, learning_rate):
    # TODO A5: zwróć nowy score.
    #
    # Pseudokod:
    # 1. weź old_score,
    # 2. pomnóż tree_correction przez learning_rate,
    # 3. dodaj wynik do old_score,
    # 4. zwróć nową wartość.
    return ...

wynik_testowy = boosting_update_student(score_start_ref, poprawka_duzo_reklamacji_ref, learning_rate_mini)

if wynik_testowy is Ellipsis:
    print("Uzupełnij funkcję boosting_update_student.")
else:
    check_close("test funkcji boosting_update_student", wynik_testowy, F1_duzo_ref)



## 2B. Jak działa boosting **na drzewach**? Trzy kolejne stump-y krok po kroku

Poniżej robimy jeszcze jeden poziom wyjaśnienia. Chodzi o to, żeby zobaczyć nie tylko wzory,
ale też **co dokładnie zapisuje się na liściach kolejnych drzew**.

Będziemy pracować na tym samym mini-zbiorze `mini`.

W tej sekcji traktujemy kolejne drzewa jako **bardzo małe stump-y** (`max_depth=1`).
Każdy stump:

1. bierze aktualne residuale,
2. dzieli obserwacje na 2 grupy,
3. wpisuje do liści poprawki,
4. aktualizuje score $F(x)$,
5. po sigmoidzie daje nowe prawdopodobieństwo.

To właśnie jest sedno Gradient Boostingu:

```text
F0
↓
T1(x)  -> pierwsza poprawka
↓
T2(x)  -> druga poprawka
↓
T3(x)  -> trzecia poprawka
↓
sigmoid(F)
↓
prawdopodobieństwo churn
```

W tej części używamy nadal intuicji dydaktycznej:

$$
r_i \approx y_i - p_i
$$

W pełnym, formalnym opisie kolejne drzewa dopasowuje się do **ujemnego gradientu funkcji straty**.
Tutaj jednak zależy nam przede wszystkim na zrozumieniu mechaniki działania modelu.



### Ręczne przeliczenie: od $F_0$ do $F_3$

#### Krok 0 — predykcja startowa

W mini-zbiorze mamy 3 klientów z churn na 8 obserwacji, więc:

$$
p_0 = \frac{3}{8} = 0.375
$$

Odpowiadający temu score startowy wynosi:

$$
F_0 = \log\left(\frac{0.375}{0.625}\right) = -0.510826
$$

Na starcie wszyscy klienci mają to samo prawdopodobieństwo churn: $0.375$.

#### Krok 1 — residuale startowe

Dydaktycznie liczymy:

$$
r_i^{(0)} = y_i - p_0
$$

Dla klasy 0 dostajemy:

$$
0 - 0.375 = -0.375
$$

Dla klasy 1 dostajemy:

$$
1 - 0.375 = 0.625
$$

Czyli na początku model:

- **za bardzo zawyża** ryzyko dla obserwacji z klasą 0,
- **za bardzo zaniża** ryzyko dla obserwacji z klasą 1.

#### Drzewo 1

Załóżmy, że pierwszy stump robi split:

```text
liczba_reklamacji >= 2 ?
```

Wartości liści liczymy jako średnie residuale w gałęziach:

- dla `liczba_reklamacji < 2`:

$$
\frac{-0.375-0.375-0.375-0.375}{4} = -0.375
$$

- dla `liczba_reklamacji >= 2`:

$$
\frac{0.625+0.625+0.625-0.375}{4} = 0.375
$$

Czyli pierwszy stump ma postać:

```text
liczba_reklamacji >= 2 ?
├── NIE  -> -0.375
└── TAK  -> +0.375
```

Przy `learning_rate = 0.8` aktualizacja ma postać:

$$
F_1(x) = F_0 + 0.8 \cdot T_1(x)
$$

Przykład:

- klient A (`reklamacje = 0`) trafia do liścia `-0.375`:

$$
F_1(A) = -0.510826 + 0.8 \cdot (-0.375) = -0.810826
$$

$$
p_1(A) = \sigma(-0.810826) \approx 0.308
$$

- klient E (`reklamacje = 2`) trafia do liścia `+0.375`:

$$
F_1(E) = -0.510826 + 0.8 \cdot 0.375 = -0.210826
$$

$$
p_1(E) = \sigma(-0.210826) \approx 0.447
$$

Nowe residuale:

$$
r_A^{(1)} = 0 - 0.308 = -0.308
$$

$$
r_E^{(1)} = 1 - 0.447 = 0.553
$$

Residuale zmalały — model poprawił się.

#### Drzewo 2

Teraz budujemy drugie drzewo już na nowych residualach.
Załóżmy, że drugi stump robi split:

```text
miesiace_umowy < 12 ?
```

Na podstawie residuali po pierwszym drzewie dostajemy wartości liści:

- dla `miesiace_umowy >= 12`:

$$
-0.3077
$$

- dla `miesiace_umowy < 12`:

$$
+0.3025
$$

Czyli:

```text
miesiace_umowy < 12 ?
├── NIE  -> -0.3077
└── TAK  -> +0.3025
```

Aktualizacja:

$$
F_2(x) = F_1(x) + 0.8 \cdot T_2(x)
$$

Dla klienta E (`miesiace_umowy = 10`):

$$
F_2(E) = -0.210826 + 0.8 \cdot 0.3025 \approx 0.0312
$$

$$
p_2(E) = \sigma(0.0312) \approx 0.508
$$

#### Drzewo 3

Załóżmy, że trzecie drzewo bierze nowe residuale i robi split:

```text
srednie_logowania_tyg < 2.25 ?
```

Dostajemy wartości liści:

- dla `srednie_logowania_tyg >= 2.25`:

$$
-0.1079
$$

- dla `srednie_logowania_tyg < 2.25`:

$$
+0.1589
$$

Czyli:

```text
srednie_logowania_tyg < 2.25 ?
├── NIE  -> -0.1079
└── TAK  -> +0.1589
```

Aktualizacja:

$$
F_3(x) = F_2(x) + 0.8 \cdot T_3(x)
$$

Dla klienta F (`logowania = 1.5`):

$$
F_3(F) = 0.0312 + 0.8 \cdot 0.1589 \approx 0.1583
$$

$$
p_3(F) = \sigma(0.1583) \approx 0.539
$$

#### Co warto zauważyć?

1. Każde kolejne drzewo wpisuje do liści **poprawki**, a nie klasy.
2. Model końcowy jest sumą małych kroków:

$$
F_3(x) = F_0 + 0.8 T_1(x) + 0.8 T_2(x) + 0.8 T_3(x)
$$

3. Późniejsze drzewa zwykle robią **mniejsze i bardziej lokalne poprawki**.
4. Jedna iteracja nie musi poprawić każdej obserwacji osobno — celem drzewa jest zmniejszenie straty **całego modelu**.


In [ ]:

# Sekcja 2B: trzy kolejne stump-y narysowane i policzone na mini-zbiorze.
# Uwaga: to umowna, dydaktyczna sekwencja drzew. Chodzi o zrozumienie mechaniki boostingu.

from IPython.display import Markdown

learning_rate_demo = 0.8


def apply_manual_stump(df, score_col, feature, operator, threshold, learning_rate=0.8):
    df = df.copy()
    df['p_current'] = sigmoid(df[score_col])
    df['residual_current'] = df['odejdzie_num'] - df['p_current']

    if operator == '>=':
        mask = df[feature] >= threshold
        yes_label = f"TAK (>= {threshold})"
        no_label = f"NIE (< {threshold})"
    elif operator == '<':
        mask = df[feature] < threshold
        yes_label = f"TAK (< {threshold})"
        no_label = f"NIE (>= {threshold})"
    else:
        raise ValueError("operator musi być '<' albo '>='")

    value_yes = df.loc[mask, 'residual_current'].mean()
    value_no = df.loc[~mask, 'residual_current'].mean()

    df['tree_value'] = np.where(mask, value_yes, value_no)
    df['score_next'] = df[score_col] + learning_rate * df['tree_value']
    df['p_next'] = sigmoid(df['score_next'])
    df['residual_next'] = df['odejdzie_num'] - df['p_next']

    summary = {
        'feature': feature,
        'operator': operator,
        'threshold': threshold,
        'yes_label': yes_label,
        'no_label': no_label,
        'value_yes': float(value_yes),
        'value_no': float(value_no),
        'mean_abs_residual_before': float(np.mean(np.abs(df['residual_current']))),
        'mean_abs_residual_after': float(np.mean(np.abs(df['residual_next']))),
    }
    return df, summary


def draw_stump(summary, title):
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.axis('off')

    root_text = f"{title}\n{summary['feature']} {summary['operator']} {summary['threshold']} ?"
    left_text = f"{summary['no_label']}\nwartość liścia = {summary['value_no']:.4f}"
    right_text = f"{summary['yes_label']}\nwartość liścia = {summary['value_yes']:.4f}"

    ax.text(0.5, 0.82, root_text, ha='center', va='center', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='black'))
    ax.text(0.22, 0.2, left_text, ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='black'))
    ax.text(0.78, 0.2, right_text, ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='black'))

    ax.annotate('', xy=(0.28, 0.31), xytext=(0.45, 0.72), arrowprops=dict(arrowstyle='->', lw=1.8))
    ax.annotate('', xy=(0.72, 0.31), xytext=(0.55, 0.72), arrowprops=dict(arrowstyle='->', lw=1.8))
    ax.text(0.36, 0.53, 'NIE', fontsize=10)
    ax.text(0.64, 0.53, 'TAK', fontsize=10)
    plt.show()


demo = mini.copy()
demo['F0_demo'] = score_start_ref
steps = [
    ('T1', 'F0_demo', 'liczba_reklamacji', '>=', 2),
    ('T2', 'F1_demo', 'miesiace_umowy', '<', 12),
    ('T3', 'F2_demo', 'srednie_logowania_tyg', '<', 2.25),
]

stump_summaries = []
trajectory_frames = []
score_col_current = 'F0_demo'
for step_name, score_col_name, feature, operator, threshold in steps:
    assert score_col_name == score_col_current
    step_df, step_summary = apply_manual_stump(demo, score_col_current, feature, operator, threshold, learning_rate_demo)
    next_score_col = step_name.replace('T', 'F')
    next_score_col = f"{next_score_col}_demo"
    demo[next_score_col] = step_df['score_next']
    stump_summaries.append((step_name, step_summary))
    trajectory_frames.append((step_name, step_df.copy(), score_col_current, next_score_col))
    score_col_current = next_score_col

summary_table = pd.DataFrame([
    {
        'drzewo': name,
        'split': f"{s['feature']} {s['operator']} {s['threshold']}",
        'wartosc_liścia_NIE': s['value_no'],
        'wartosc_liścia_TAK': s['value_yes'],
        'mean_|residual|_przed': s['mean_abs_residual_before'],
        'mean_|residual|_po': s['mean_abs_residual_after'],
    }
    for name, s in stump_summaries
])

display(summary_table.round(4))
for name, summary in stump_summaries:
    draw_stump(summary, title=name)

trajectory = pd.DataFrame({'klient': demo['klient'], 'y': demo['odejdzie_num']})
trajectory['F0'] = demo['F0_demo']
trajectory['p0'] = sigmoid(demo['F0_demo'])
for k in [1, 2, 3]:
    trajectory[f'F{k}'] = demo[f'F{k}_demo']
    trajectory[f'p{k}'] = sigmoid(demo[f'F{k}_demo'])
    trajectory[f'residual_{k}'] = trajectory['y'] - trajectory[f'p{k}']
trajectory['residual_0'] = trajectory['y'] - trajectory['p0']
cols = ['klient', 'y', 'F0', 'p0', 'residual_0', 'F1', 'p1', 'residual_1', 'F2', 'p2', 'residual_2', 'F3', 'p3', 'residual_3']
display(trajectory[cols].round(3))

selected_clients = trajectory[trajectory['klient'].isin(['A', 'E', 'F', 'H'])].copy()
display(selected_clients[cols].round(3))



### Pseudokod algorytmu Gradient Boosting (wersja intuicyjna)

```text
Wejście:
- dane treningowe (x_i, y_i)
- liczba drzew M
- learning_rate = eta

1. Ustal model startowy F0.
   W klasyfikacji binarnej można myśleć o nim jako o stałym score,
   odpowiadającym bazowemu prawdopodobieństwu klasy 1.

2. Dla m = 1, 2, ..., M:
   a. policz aktualne prawdopodobieństwa p_i = sigmoid(F_{m-1}(x_i))
   b. policz pseudo-residuale / kierunek poprawki
   c. dopasuj małe drzewo T_m do tych residuali
   d. odczytaj z liścia poprawkę T_m(x_i)
   e. zaktualizuj score:

      F_m(x) = F_{m-1}(x) + eta * T_m(x)

3. Na końcu zamień score na prawdopodobieństwo:

   p(x) = sigmoid(F_M(x))

4. Jeśli trzeba, zastosuj próg decyzyjny i zamień prawdopodobieństwo na klasę.
```



### Ćwiczenie A6: kontynuacja — spróbuj samodzielnie przejść przez kolejne iteracje

Poniżej masz uproszczone zadanie „ciąg dalszy”. Chodzi o to, żeby przećwiczyć logikę:

```text
residuale -> liść drzewa -> poprawka -> nowy score -> nowe prawdopodobieństwo
```

Wykorzystaj wyniki z wcześniejszych obliczeń oraz dane z sekcji 2B.


In [ ]:

# Ćwiczenie A6 — wersja studencka.
#
# Załóżmy, że po pierwszym drzewie budujemy drugi stump:
#     miesiace_umowy < 12 ?
# i z obliczeń dla residuali po pierwszym drzewie dostajemy:
#     liść NIE (>=12): -0.3077
#     liść TAK (<12):  +0.3025
#
# Przyjmij learning_rate = 0.8.
#
# TODO A6.1:
# Dla klienta E (miesiace_umowy = 10) policz ręcznie:
# 1. F2_E = F1_E + 0.8 * 0.3025
# 2. p2_E = sigmoid(F2_E)
#
# TODO A6.2:
# Dla klienta A (miesiace_umowy = 60) policz ręcznie:
# 1. F2_A = F1_A + 0.8 * (-0.3077)
# 2. p2_A = sigmoid(F2_A)
#
# TODO A6.3:
# Załóżmy teraz trzecie drzewo:
#     srednie_logowania_tyg < 2.25 ?
# z wartościami liści:
#     liść NIE (>=2.25): -0.1079
#     liść TAK (<2.25):  +0.1589
#
# Dla klienta F (logowania = 1.5) policz:
# 1. F3_F = F2_F + 0.8 * 0.1589
# 2. p3_F = sigmoid(F3_F)

F1_E_recznie = ...
p2_E_recznie = ...
F2_A_recznie = ...
p2_A_recznie = ...
F3_F_recznie = ...
p3_F_recznie = ...

# Wartości referencyjne policzone kodem z sekcji 2B.
selected = trajectory.set_index('klient')
F1_E_ref = float(selected.loc['E', 'F1'])
p2_E_ref = float(selected.loc['E', 'p2'])
F2_A_ref = float(selected.loc['A', 'F2'])
p2_A_ref = float(selected.loc['A', 'p2'])
F3_F_ref = float(selected.loc['F', 'F3'])
p3_F_ref = float(selected.loc['F', 'p3'])

check_close('A6: F1_E (kontrola: score po pierwszym drzewie dla E)', F1_E_recznie, F1_E_ref)
check_close('A6: p2_E', p2_E_recznie, p2_E_ref)
check_close('A6: F2_A', F2_A_recznie, F2_A_ref)
check_close('A6: p2_A', p2_A_recznie, p2_A_ref)
check_close('A6: F3_F', F3_F_recznie, F3_F_ref)
check_close('A6: p3_F', p3_F_recznie, p3_F_ref)



### Najważniejsza intuicja po sekcji 2B

- **Random Forest** buduje wiele drzew niezależnie i potem je uśrednia / daje im głosowanie.
- **Gradient Boosting** buduje małe drzewa **sekwencyjnie**.
- Liść drzewa w boostingu nie oznacza „klasa tak/nie”, tylko **poprawkę do score**.
- Model końcowy to suma wielu małych kroków.
- Dlatego pojedyncze drzewo w boostingu może być bardzo proste, a mimo to cały model może być silny.



## 2C. Co zostaje po treningu boostingu i jak model przewiduje nowego klienta?

Po poprzedniej sekcji łatwo zadać bardzo ważne pytanie:

```text
Skoro mamy T1, T2, T3, ..., to które drzewo wybieramy na końcu?
```

Odpowiedź brzmi: **nie wybieramy jednego drzewa**.

Gradient Boosting nie działa tak, że jedno z drzew zostaje uznane za „najlepsze”.
Po treningu zostaje cały model:

$$
F_M(x) = F_0 + \eta T_1(x) + \eta T_2(x) + \eta T_3(x) + \ldots + \eta T_M(x)
$$

czyli:

```text
predykcja startowa
+ poprawka z drzewa 1
+ poprawka z drzewa 2
+ poprawka z drzewa 3
+ ...
```

Dopiero na końcu zamieniamy score na prawdopodobieństwo:

$$
p_M(x)=\sigma(F_M(x))
$$

Jeżeli chcemy dostać klasę `odejdzie` / `nie odejdzie`, to później dokładamy jeszcze próg decyzyjny, np. 0.5 albo próg dobrany biznesowo.



### Czy każde kolejne drzewo może używać innej cechy?

Tak. Każde kolejne drzewo patrzy na **aktualne residuale**, czyli na to, czego model jeszcze dobrze nie wyjaśnia.
Nie ma obowiązku używania tej samej cechy co wcześniej.

W naszej demonstracji mamy:

| Iteracja | Split | Na czym pracuje to drzewo? | Intuicja |
|---|---|---|---|
| $T_1$ | `liczba_reklamacji >= 2` | residuale po modelu startowym $F_0$ | najpierw reklamacje dobrze rozdzielają część klientów |
| $T_2$ | `miesiace_umowy < 12` | residuale po $F_0 + \eta T_1$ | po pierwszej poprawce zostały błędy, które lepiej tłumaczy krótka umowa |
| $T_3$ | `srednie_logowania_tyg < 2.25` | residuale po $F_0 + \eta T_1 + \eta T_2$ | potem część pozostałych błędów lepiej tłumaczy niska aktywność |

W prawdziwym treningu algorytm w każdej iteracji sprawdza możliwe splity i wybiera ten, który najlepiej zmniejsza aktualną stratę.
Czasem kolejne drzewo użyje nowej cechy, a czasem wróci do tej samej cechy z innym progiem.

Ważne jest to, że kolejne drzewa nie tworzą jednego większego drzewa. To są osobne drzewa, których wyniki się sumują.



### Przykład: jak model przewiduje nowego klienta?

Weźmy nowego klienta:

| cecha | wartość |
|---|---:|
| `liczba_reklamacji` | 3 |
| `miesiace_umowy` | 8 |
| `srednie_logowania_tyg` | 1.5 |

Startujemy od:

$$
F_0=-0.510826
$$

#### Przejście przez $T_1$

Pierwsze drzewo:

```text
liczba_reklamacji >= 2 ?
```

Nowy klient ma `liczba_reklamacji = 3`, więc idzie do gałęzi `TAK` i dostaje poprawkę:

$$
T_1(x)=0.375
$$

Aktualizacja:

$$
F = -0.510826 + 0.8 \cdot 0.375 = -0.210826
$$

#### Przejście przez $T_2$

Drugie drzewo:

```text
miesiace_umowy < 12 ?
```

Nowy klient ma `miesiace_umowy = 8`, więc idzie do gałęzi `TAK` i dostaje poprawkę:

$$
T_2(x)=0.3025
$$

Aktualizacja:

$$
F = -0.210826 + 0.8 \cdot 0.3025 \approx 0.0312
$$

#### Przejście przez $T_3$

Trzecie drzewo:

```text
srednie_logowania_tyg < 2.25 ?
```

Nowy klient ma `srednie_logowania_tyg = 1.5`, więc idzie do gałęzi `TAK` i dostaje poprawkę:

$$
T_3(x)=0.1589
$$

Aktualizacja:

$$
F = 0.0312 + 0.8 \cdot 0.1589 \approx 0.1583
$$

Na końcu liczymy sigmoid:

$$
p = \sigma(0.1583) \approx 0.539
$$

Czyli w tej uproszczonej demonstracji model zwraca około:

```text
53.9% prawdopodobieństwa churn
```

To nie jest wynik jednego drzewa. To jest wynik **sumy poprawek ze wszystkich drzew**.


In [ ]:

# Sekcja 2C: predykcja nowego klienta przez cały mini-model boostingowy.
# Używamy trzech stumpów z sekcji 2B.

manual_stumps = []
for step_name, summary in stump_summaries:
    manual_stumps.append({
        'drzewo': step_name,
        'feature': summary['feature'],
        'operator': summary['operator'],
        'threshold': summary['threshold'],
        'value_yes': summary['value_yes'],
        'value_no': summary['value_no'],
    })


def _goes_yes(observation, stump):
    value = observation[stump['feature']]
    if stump['operator'] == '>=':
        return value >= stump['threshold']
    if stump['operator'] == '<':
        return value < stump['threshold']
    raise ValueError("operator musi być '<' albo '>='")


def predict_manual_boosting(observation, base_score=score_start_ref, learning_rate=learning_rate_demo):
    """Przepuszcza jedną obserwację przez wszystkie ręcznie zdefiniowane stump-y."""
    score = float(base_score)
    rows = [{
        'krok': 'F0',
        'pytanie': 'model startowy',
        'odpowiedz': '-',
        'poprawka_z_liścia': 0.0,
        'score_po_kroku': score,
        'p_po_kroku': float(sigmoid(score)),
    }]

    for stump in manual_stumps:
        yes = _goes_yes(observation, stump)
        leaf_value = stump['value_yes'] if yes else stump['value_no']
        score = score + learning_rate * leaf_value
        question = f"{stump['feature']} {stump['operator']} {stump['threshold']}?"
        rows.append({
            'krok': stump['drzewo'],
            'pytanie': question,
            'odpowiedz': 'TAK' if yes else 'NIE',
            'poprawka_z_liścia': leaf_value,
            'score_po_kroku': score,
            'p_po_kroku': float(sigmoid(score)),
        })

    return pd.DataFrame(rows)


def draw_prediction_flow(path_table, title='Przejście jednej obserwacji przez boosting'):
    labels = []
    for _, row in path_table.iterrows():
        if row['krok'] == 'F0':
            labels.append(
                f"F0: model startowy\nscore = {row['score_po_kroku']:.4f}\np = {row['p_po_kroku']:.3f}"
            )
        else:
            labels.append(
                f"{row['krok']}: {row['pytanie']}\n"
                f"odpowiedź: {row['odpowiedz']}\n"
                f"liść = {row['poprawka_z_liścia']:.4f}\n"
                f"score = {row['score_po_kroku']:.4f}\n"
                f"p = {row['p_po_kroku']:.3f}"
            )

    fig, ax = plt.subplots(figsize=(9, 2.1 * len(labels)))
    ax.axis('off')
    y_positions = np.linspace(0.9, 0.1, len(labels))
    for i, (label, y) in enumerate(zip(labels, y_positions)):
        ax.text(0.5, y, label, ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.45', fc='white', ec='black'))
        if i < len(labels) - 1:
            ax.annotate('', xy=(0.5, y_positions[i+1] + 0.055), xytext=(0.5, y - 0.055),
                        arrowprops=dict(arrowstyle='->', lw=1.5))
    ax.set_title(title, fontsize=12)
    plt.show()


nowy_klient = {
    'liczba_reklamacji': 3,
    'miesiace_umowy': 8,
    'srednie_logowania_tyg': 1.5,
}

path_nowy = predict_manual_boosting(nowy_klient)
display(path_nowy.round(4))
draw_prediction_flow(path_nowy, title='Nowy klient: reklamacje=3, umowa=8 mies., logowania=1.5')

final_probability = path_nowy.iloc[-1]['p_po_kroku']
print(f"Końcowe prawdopodobieństwo churn: {final_probability:.3f}")



### Kiedy kończymy budować kolejne drzewa?

W tym mini-przykładzie widzimy, że po kolejnych poprawkach średni błąd maleje:

```text
F0  -> duże residuale
T1  -> residuale mniejsze
T2  -> residuale jeszcze mniejsze
T3  -> kolejna, mniejsza poprawka
```

W prawdziwym modelu zwykle nie patrzymy tylko na residuale treningowe, bo model może zacząć się przeuczać.
Dlatego liczbę drzew kontroluje się przez:

- `n_estimators`, czyli maksymalną liczbę drzew,
- `learning_rate`, czyli wielkość pojedynczego kroku,
- wynik na zbiorze walidacyjnym,
- czasem `early_stopping`, czyli zatrzymanie, gdy kolejne drzewa nie poprawiają już walidacji.

Nie wybieramy więc „najlepszego pojedynczego drzewa”.
Wybieramy raczej **ile kolejnych poprawek** chcemy dodać do modelu.


In [ ]:

# Mini-podsumowanie: jak zmienia się średni bezwzględny residual po kolejnych stumpach.
# To jest metryka dydaktyczna dla małego przykładu, nie główna metryka modelu produkcyjnego.

progress_rows = [{
    'etap': 'F0: model startowy',
    'liczba_drzew': 0,
    'mean_abs_residual': float(summary_table.loc[0, 'mean_|residual|_przed']),
}]

for idx, row in summary_table.iterrows():
    progress_rows.append({
        'etap': f"po {row['drzewo']}",
        'liczba_drzew': int(idx + 1),
        'mean_abs_residual': float(row['mean_|residual|_po']),
    })

progress_demo = pd.DataFrame(progress_rows)
display(progress_demo.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(progress_demo['liczba_drzew'], progress_demo['mean_abs_residual'], marker='o')
ax.set_xlabel('Liczba dodanych stumpów')
ax.set_ylabel('Średni |residual|')
ax.set_title('Mini-przykład: kolejne drzewa zmniejszają pozostały błąd')
ax.set_xticks(progress_demo['liczba_drzew'])
plt.show()



### Ćwiczenie A7: przewidź ręcznie nowego klienta przez wszystkie trzy stump-y

Teraz najważniejsze pytanie praktyczne:

```text
Jak gotowy boosting przewiduje nową obserwację?
```

Policz ręcznie wynik dla klienta:

| cecha | wartość |
|---|---:|
| `liczba_reklamacji` | 0 |
| `miesiace_umowy` | 60 |
| `srednie_logowania_tyg` | 6.0 |

Użyj tych samych trzech drzew:

```text
T1: liczba_reklamacji >= 2 ?
T2: miesiace_umowy < 12 ?
T3: srednie_logowania_tyg < 2.25 ?
```

oraz wzoru:

$$
F_{nowe} = F_{stare} + learning\_rate \cdot poprawka
$$

Na końcu policz:

$$
p = \sigma(F)
$$


In [ ]:

# Ćwiczenie A7 — wersja studencka.
#
# TODO: dla klienta:
#     liczba_reklamacji = 0
#     miesiace_umowy = 60
#     srednie_logowania_tyg = 6.0
#
# wpisz:
# 1. poprawkę z T1,
# 2. poprawkę z T2,
# 3. poprawkę z T3,
# 4. końcowy score F3,
# 5. końcowe prawdopodobieństwo p3.
#
# Podpowiedź:
# - ten klient powinien iść w każdym z trzech drzew gałęzią NIE.

A7_poprawka_T1 = ...
A7_poprawka_T2 = ...
A7_poprawka_T3 = ...
A7_F3 = ...
A7_p3 = ...

klient_A7 = {
    'liczba_reklamacji': 0,
    'miesiace_umowy': 60,
    'srednie_logowania_tyg': 6.0,
}
path_A7_ref = predict_manual_boosting(klient_A7)

A7_poprawka_T1_ref = float(path_A7_ref.loc[path_A7_ref['krok'] == 'T1', 'poprawka_z_liścia'].iloc[0])
A7_poprawka_T2_ref = float(path_A7_ref.loc[path_A7_ref['krok'] == 'T2', 'poprawka_z_liścia'].iloc[0])
A7_poprawka_T3_ref = float(path_A7_ref.loc[path_A7_ref['krok'] == 'T3', 'poprawka_z_liścia'].iloc[0])
A7_F3_ref = float(path_A7_ref.iloc[-1]['score_po_kroku'])
A7_p3_ref = float(path_A7_ref.iloc[-1]['p_po_kroku'])

check_close('A7: poprawka T1', A7_poprawka_T1, A7_poprawka_T1_ref)
check_close('A7: poprawka T2', A7_poprawka_T2, A7_poprawka_T2_ref)
check_close('A7: poprawka T3', A7_poprawka_T3, A7_poprawka_T3_ref)
check_close('A7: końcowy score F3', A7_F3, A7_F3_ref)
check_close('A7: końcowe prawdopodobieństwo p3', A7_p3, A7_p3_ref)



## 2D. Mały `learning_rate` i wiele iteracji — domknięcie boostingu

W poprzednich ręcznych rachunkach używaliśmy:

```python
learning_rate = 0.8
```

To jest celowo duża wartość, żeby po 1–3 drzewach było wyraźnie widać zmianę prawdopodobieństw.

W praktyce często używa się mniejszych wartości, np.:

```python
learning_rate = 0.1
learning_rate = 0.05
learning_rate = 0.03
```

Wtedy pojedyncze drzewo robi tylko mały krok:

$$
F_{nowe}(x)=F_{stare}(x)+\eta T_m(x)
$$

Jeżeli $\eta$ jest mniejsze, to:

- predykcje zmieniają się spokojniej,
- mniejsze jest ryzyko „przeskoczenia” za mocno w jedną stronę,
- ale potrzeba więcej drzew, żeby uzyskać podobny efekt.

To jest klasyczny kompromis:

```text
duży learning_rate + mało drzew
    szybciej, ale bardziej skokowo

mały learning_rate + dużo drzew
    wolniej, ale zwykle stabilniej
```



### Te same trzy stump-y: `learning_rate = 0.8` vs `learning_rate = 0.1`

Najpierw porównajmy ten sam schemat trzech splitów:

```text
T1: liczba_reklamacji >= 2
T2: miesiace_umowy < 12
T3: srednie_logowania_tyg < 2.25
```

W obu przypadkach po każdym drzewie liczymy nowe residuale i nowe wartości liści.
Różni się tylko wielkość kroku, czyli `learning_rate`.

Dzięki temu widać, że przy `0.1` kierunek zmian jest ten sam, ale ruch jest dużo wolniejszy.


In [ ]:

# Porównanie: duży learning_rate 0.8 i bardziej realistyczny learning_rate 0.1.
# Używamy tej samej sekwencji splitów co w ręcznej demonstracji,
# ale wartości liści przeliczamy po każdym kroku dla danego learning_rate.

fixed_split_specs = [
    ('T1', 'liczba_reklamacji', '>=', 2),
    ('T2', 'miesiace_umowy', '<', 12),
    ('T3', 'srednie_logowania_tyg', '<', 2.25),
]


def _split_mask_for_operator(df_or_obs, feature, operator, threshold):
    """Maska dla DataFrame albo odpowiedź dla pojedynczej obserwacji."""
    if isinstance(df_or_obs, pd.DataFrame):
        values = df_or_obs[feature]
    else:
        values = df_or_obs[feature]

    if operator == '>=':
        return values >= threshold
    if operator == '<':
        return values < threshold
    raise ValueError("operator musi być '<' albo '>='")


def run_fixed_split_sequence(df, split_specs, learning_rate, base_score=score_start_ref):
    """Uruchamia zadaną sekwencję stumpów i przelicza wartości liści po każdej iteracji."""
    y = df['odejdzie_num'].to_numpy(dtype=float)
    scores = np.repeat(float(base_score), len(df))
    rows = []
    model_steps = []

    p = sigmoid(scores)
    rows.append({
        'learning_rate': learning_rate,
        'iteracja': 0,
        'split': 'F0',
        'leaf_NIE': 0.0,
        'leaf_TAK': 0.0,
        'log_loss': float(log_loss(y, p, labels=[0, 1])),
        'mean_abs_residual': float(np.mean(np.abs(y - p))),
        **{f"p_{k}": float(p_i) for k, p_i in zip(df['klient'], p)}
    })

    for iteration, (tree_name, feature, operator, threshold) in enumerate(split_specs, start=1):
        p = sigmoid(scores)
        residual = y - p
        mask_yes = _split_mask_for_operator(df, feature, operator, threshold).to_numpy()

        leaf_yes = float(residual[mask_yes].mean())
        leaf_no = float(residual[~mask_yes].mean())
        tree_values = np.where(mask_yes, leaf_yes, leaf_no)

        scores = scores + learning_rate * tree_values
        p_new = sigmoid(scores)

        model_steps.append({
            'drzewo': tree_name,
            'feature': feature,
            'operator': operator,
            'threshold': threshold,
            'leaf_TAK': leaf_yes,
            'leaf_NIE': leaf_no,
        })

        rows.append({
            'learning_rate': learning_rate,
            'iteracja': iteration,
            'split': f"{tree_name}: {feature} {operator} {threshold}",
            'leaf_NIE': leaf_no,
            'leaf_TAK': leaf_yes,
            'log_loss': float(log_loss(y, p_new, labels=[0, 1])),
            'mean_abs_residual': float(np.mean(np.abs(y - p_new))),
            **{f"p_{k}": float(p_i) for k, p_i in zip(df['klient'], p_new)}
        })

    return pd.DataFrame(rows), model_steps


def predict_from_fixed_sequence(observation, model_steps, learning_rate, base_score=score_start_ref):
    """Predykcja jednej obserwacji przez sekwencję stumpów z przeliczonymi liśćmi."""
    score = float(base_score)
    rows = [{
        'iteracja': 0,
        'drzewo': 'F0',
        'odpowiedz': '-',
        'leaf_value': 0.0,
        'score': score,
        'p': float(sigmoid(score)),
    }]

    for step in model_steps:
        goes_yes = bool(_split_mask_for_operator(observation, step['feature'], step['operator'], step['threshold']))
        leaf_value = step['leaf_TAK'] if goes_yes else step['leaf_NIE']
        score = score + learning_rate * leaf_value
        rows.append({
            'iteracja': len(rows),
            'drzewo': step['drzewo'],
            'odpowiedz': 'TAK' if goes_yes else 'NIE',
            'leaf_value': leaf_value,
            'score': score,
            'p': float(sigmoid(score)),
        })

    return pd.DataFrame(rows)


fixed_histories = {}
fixed_models = {}

for lr in [0.8, 0.1]:
    hist_lr, model_lr = run_fixed_split_sequence(mini, fixed_split_specs, learning_rate=lr)
    fixed_histories[lr] = hist_lr
    fixed_models[lr] = model_lr

compare_fixed_lr = pd.concat(
    [
        fixed_histories[0.8][['learning_rate', 'iteracja', 'split', 'leaf_NIE', 'leaf_TAK', 'mean_abs_residual', 'p_A', 'p_E', 'p_H']],
        fixed_histories[0.1][['learning_rate', 'iteracja', 'split', 'leaf_NIE', 'leaf_TAK', 'mean_abs_residual', 'p_A', 'p_E', 'p_H']],
    ],
    ignore_index=True,
)

display(compare_fixed_lr.round(4))

# Dwa nowi klienci testowi:
# - wysokie_ryzyko: dużo reklamacji, krótka umowa, mało logowań
# - niskie_ryzyko: brak reklamacji, długa umowa, dużo logowań
client_high_risk = {
    'liczba_reklamacji': 3,
    'miesiace_umowy': 8,
    'srednie_logowania_tyg': 1.5,
}

client_low_risk = {
    'liczba_reklamacji': 0,
    'miesiace_umowy': 60,
    'srednie_logowania_tyg': 6.0,
}

lr_client_rows = []
for label, obs in [('wysokie_ryzyko', client_high_risk), ('niskie_ryzyko', client_low_risk)]:
    for lr in [0.8, 0.1]:
        path = predict_from_fixed_sequence(obs, fixed_models[lr], learning_rate=lr)
        lr_client_rows.append({
            'klient_demo': label,
            'learning_rate': lr,
            'p_start': path.loc[path['iteracja'] == 0, 'p'].iloc[0],
            'p_po_T1': path.loc[path['iteracja'] == 1, 'p'].iloc[0],
            'p_po_T2': path.loc[path['iteracja'] == 2, 'p'].iloc[0],
            'p_po_T3': path.loc[path['iteracja'] == 3, 'p'].iloc[0],
        })

lr_new_client_compare = pd.DataFrame(lr_client_rows)
display(lr_new_client_compare.round(4))



### Więcej iteracji przy `learning_rate = 0.1`

Po 3 stumpach z `learning_rate = 0.1` predykcje zmieniły się tylko trochę.
To nie znaczy, że mały `learning_rate` jest zły. To znaczy, że potrzebujemy więcej iteracji.

Teraz zrobimy prostą, dydaktyczną pętlę boostingu:

```text
dla każdej iteracji:
    policz aktualne residuale
    sprawdź możliwe stump-y
    wybierz stump, który najbardziej poprawia log-loss
    dodaj małą poprawkę z learning_rate = 0.1
```

To nadal nie jest pełna implementacja XGBoost ani scikit-learn, ale dobrze pokazuje mechanikę:

```text
mały krok + wiele powtórzeń = stopniowe dochodzenie do mocniejszej predykcji
```

Zwróć uwagę na jeszcze jedną rzecz: algorytm może wybrać tę samą cechę wiele razy.
To jest normalne. W każdej iteracji pyta po prostu:

```text
który split najlepiej tłumaczy aktualny pozostały błąd?
```


In [ ]:

# Dydaktyczna pętla boostingu z wieloma stumpami i learning_rate = 0.1.
# W każdej iteracji wybieramy split, który po małym kroku daje najniższy log-loss.

def candidate_thresholds(values):
    """Progi między kolejnymi unikalnymi wartościami cechy."""
    unique_values = sorted(pd.Series(values).dropna().unique())
    return [(a + b) / 2 for a, b in zip(unique_values[:-1], unique_values[1:])]


def run_greedy_residual_boosting(df, n_iterations=40, learning_rate=0.1, base_score=score_start_ref):
    y = df['odejdzie_num'].to_numpy(dtype=float)
    features = ['liczba_reklamacji', 'miesiace_umowy', 'srednie_logowania_tyg']
    scores = np.repeat(float(base_score), len(df))

    history_rows = []
    model_steps = []

    p0 = sigmoid(scores)
    history_rows.append({
        'iteracja': 0,
        'split': 'F0',
        'leaf_NIE': 0.0,
        'leaf_TAK': 0.0,
        'log_loss': float(log_loss(y, p0, labels=[0, 1])),
        'mean_abs_residual': float(np.mean(np.abs(y - p0))),
        **{f"p_{k}": float(p_i) for k, p_i in zip(df['klient'], p0)}
    })

    for iteration in range(1, n_iterations + 1):
        p = sigmoid(scores)
        residual = y - p

        best = None

        for feature in features:
            for threshold in candidate_thresholds(df[feature]):
                mask_yes = (df[feature].to_numpy() >= threshold)
                if mask_yes.sum() == 0 or (~mask_yes).sum() == 0:
                    continue

                leaf_yes = float(residual[mask_yes].mean())
                leaf_no = float(residual[~mask_yes].mean())
                tree_values = np.where(mask_yes, leaf_yes, leaf_no)

                candidate_scores = scores + learning_rate * tree_values
                candidate_p = sigmoid(candidate_scores)
                candidate_log_loss = float(log_loss(y, candidate_p, labels=[0, 1]))

                candidate = {
                    'feature': feature,
                    'threshold': threshold,
                    'leaf_TAK': leaf_yes,
                    'leaf_NIE': leaf_no,
                    'tree_values': tree_values,
                    'scores_after': candidate_scores,
                    'p_after': candidate_p,
                    'log_loss_after': candidate_log_loss,
                    'mean_abs_residual_after': float(np.mean(np.abs(y - candidate_p))),
                }

                if best is None or candidate['log_loss_after'] < best['log_loss_after']:
                    best = candidate

        scores = best['scores_after']
        p_after = best['p_after']

        model_steps.append({
            'feature': best['feature'],
            'operator': '>=',
            'threshold': best['threshold'],
            'leaf_TAK': best['leaf_TAK'],
            'leaf_NIE': best['leaf_NIE'],
        })

        history_rows.append({
            'iteracja': iteration,
            'split': f"{best['feature']} >= {best['threshold']:.3g}",
            'leaf_NIE': best['leaf_NIE'],
            'leaf_TAK': best['leaf_TAK'],
            'log_loss': best['log_loss_after'],
            'mean_abs_residual': best['mean_abs_residual_after'],
            **{f"p_{k}": float(p_i) for k, p_i in zip(df['klient'], p_after)}
        })

    return pd.DataFrame(history_rows), model_steps


history_lr_01_many, greedy_stumps_lr_01 = run_greedy_residual_boosting(
    mini,
    n_iterations=40,
    learning_rate=0.1,
)

interesting_iterations = [0, 1, 2, 3, 5, 10, 20, 30, 40]

display(
    history_lr_01_many.loc[
        history_lr_01_many['iteracja'].isin(interesting_iterations),
        ['iteracja', 'split', 'leaf_NIE', 'leaf_TAK', 'log_loss', 'mean_abs_residual', 'p_A', 'p_E', 'p_F', 'p_H']
    ].round(4)
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_lr_01_many['iteracja'], history_lr_01_many['log_loss'], marker='o')
ax.set_xlabel('Iteracja / liczba stumpów')
ax.set_ylabel('Log-loss na mini-zbiorze')
ax.set_title('Mały learning_rate: strata maleje stopniowo')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for klient in ['A', 'E', 'F', 'H']:
    ax.plot(history_lr_01_many['iteracja'], history_lr_01_many[f'p_{klient}'], marker='o', label=f'klient {klient}')
ax.set_xlabel('Iteracja / liczba stumpów')
ax.set_ylabel('Przewidywane prawdopodobieństwo churn')
ax.set_title('Jak ewoluują prawdopodobieństwa dla wybranych klientów')
ax.legend()
plt.show()


def predict_from_greedy_stumps(observation, model_steps, learning_rate=0.1, base_score=score_start_ref):
    score = float(base_score)
    rows = [{
        'iteracja': 0,
        'split': 'F0',
        'odpowiedz': '-',
        'leaf_value': 0.0,
        'score': score,
        'p': float(sigmoid(score)),
    }]

    for iteration, step in enumerate(model_steps, start=1):
        goes_yes = observation[step['feature']] >= step['threshold']
        leaf_value = step['leaf_TAK'] if goes_yes else step['leaf_NIE']
        score = score + learning_rate * leaf_value
        rows.append({
            'iteracja': iteration,
            'split': f"{step['feature']} >= {step['threshold']:.3g}",
            'odpowiedz': 'TAK' if goes_yes else 'NIE',
            'leaf_value': leaf_value,
            'score': score,
            'p': float(sigmoid(score)),
        })

    return pd.DataFrame(rows)


path_high_many = predict_from_greedy_stumps(client_high_risk, greedy_stumps_lr_01, learning_rate=0.1)
path_low_many = predict_from_greedy_stumps(client_low_risk, greedy_stumps_lr_01, learning_rate=0.1)

new_client_many_iters = pd.concat(
    [
        path_high_many.assign(klient_demo='wysokie_ryzyko'),
        path_low_many.assign(klient_demo='niskie_ryzyko'),
    ],
    ignore_index=True,
)

display(
    new_client_many_iters.loc[
        new_client_many_iters['iteracja'].isin(interesting_iterations),
        ['klient_demo', 'iteracja', 'split', 'odpowiedz', 'leaf_value', 'score', 'p']
    ].round(4)
)



### Ćwiczenie A8: mały `learning_rate` i więcej drzew

Na podstawie tabel powyżej uzupełnij trzy wartości.

Pytania:

1. Jakie prawdopodobieństwo churn ma klient `wysokie_ryzyko` po 3 stumpach przy `learning_rate = 0.8`?
2. Jakie prawdopodobieństwo churn ma ten sam klient po 3 stumpach przy `learning_rate = 0.1`?
3. W wieloiteracyjnej demonstracji z `learning_rate = 0.1`: po ilu stumpach klient `wysokie_ryzyko` pierwszy raz przekracza prawdopodobieństwo `0.5`?

To ćwiczenie ma pokazać intuicję:

```text
0.1 nie oznacza słabszego modelu.
0.1 oznacza mniejsze kroki, więc potrzeba więcej kroków.
```


In [ ]:

# Ćwiczenie A8 — wersja studencka.
#
# TODO:
# Odczytaj wartości z tabel powyżej albo policz je kodem.
#
# Podpowiedź:
# - tabela lr_new_client_compare pokazuje wyniki po 3 ręcznych stumpach
#   dla learning_rate 0.8 oraz 0.1,
# - tabela new_client_many_iters pokazuje, jak wynik zmienia się przy wielu iteracjach.

A8_p_high_lr_08_po_T3 = ...
A8_p_high_lr_01_po_T3 = ...
A8_pierwsza_iteracja_powyzej_050 = ...

A8_p_high_lr_08_po_T3_ref = float(
    lr_new_client_compare.query("klient_demo == 'wysokie_ryzyko' and learning_rate == 0.8")['p_po_T3'].iloc[0]
)
A8_p_high_lr_01_po_T3_ref = float(
    lr_new_client_compare.query("klient_demo == 'wysokie_ryzyko' and learning_rate == 0.1")['p_po_T3'].iloc[0]
)
A8_pierwsza_iteracja_powyzej_050_ref = int(
    path_high_many.loc[path_high_many['p'] >= 0.5, 'iteracja'].iloc[0]
)

check_close('A8: p wysokie_ryzyko po T3, lr=0.8', A8_p_high_lr_08_po_T3, A8_p_high_lr_08_po_T3_ref, atol=1e-4)
check_close('A8: p wysokie_ryzyko po T3, lr=0.1', A8_p_high_lr_01_po_T3, A8_p_high_lr_01_po_T3_ref, atol=1e-4)
check_close('A8: pierwsza iteracja p >= 0.5 przy lr=0.1', A8_pierwsza_iteracja_powyzej_050, A8_pierwsza_iteracja_powyzej_050_ref, atol=0)



### Co domyka ta część?

Po tej sekcji cały boosting można streścić tak:

```text
1. Startujemy od F0.
2. Liczymy aktualne błędy / gradienty.
3. Budujemy małe drzewo-poprawkę.
4. Dodajemy tylko część poprawki: eta * T_m(x).
5. Powtarzamy wiele razy.
6. Na końcu używamy całej sumy drzew, nie jednego wybranego drzewa.
7. Zamieniamy końcowy score na prawdopodobieństwo przez sigmoid.
```

Najważniejsze zdanie:

> Gradient Boosting to nie głosowanie drzew.  
> To suma wielu małych, kolejnych poprawek.



### Most do XGBoost

Po tym przykładzie łatwiej zrozumieć XGBoost.

Architektura zostaje bardzo podobna:

$$
F(x) = F_0 + \eta T_1(x) + \eta T_2(x) + \ldots
$$

Zmienia się głównie to, **jak liczymy najlepsze splity i wartości liści**.

W naszym uproszczeniu wartość liścia była średnim residualem.
W XGBoost wartość liścia jest liczona z gradientów, Hessianów i regularizacji.

Czyli pytanie nie brzmi już tylko:

```text
jaka jest średnia poprawka w tym liściu?
```

ale raczej:

```text
jaka poprawka najbardziej zmniejsza stratę,
ale nie jest zbyt agresywna po uwzględnieniu regularizacji?
```


## 3. Co XGBoost dodaje do Gradient Boostingu?

XGBoost to nie „zupełnie inny pomysł” niż Gradient Boosting. To bardzo dopracowana wersja boostingu drzewiastego.

Najważniejsze dodatki:

1. **gradient i Hessian** — XGBoost używa informacji pierwszego i drugiego rzędu o funkcji straty,
2. **regularizacja liści** — kara za zbyt duże wartości liści i zbyt złożone drzewa,
3. **kontrola złożoności** — np. `max_depth`, `min_child_weight`, `gamma`, `reg_lambda`,
4. **losowanie obserwacji i cech** — `subsample`, `colsample_bytree`, trochę podobne do idei z Random Forest,
5. **wydajna implementacja** — szybkie budowanie drzew, histogramy, obsługa braków danych.

Dla klasyfikacji binarnej z log-loss możemy zapamiętać intuicję:

```text
gradient g = p - y
Hessian  h = p * (1 - p)
```

Wartość liścia w XGBoost, w uproszczonym zapisie, wygląda tak:

$$
w = -\frac{\sum g_i}{\sum h_i + \lambda}
$$

Czyli XGBoost pyta:

```text
Jaka poprawka w tym liściu najbardziej zmniejsza stratę,
ale nie jest zbyt agresywna dzięki regularizacji lambda?
```


W praktyce XGBoost nie pyta tylko: „czy split poprawia dopasowanie?”. Pyta raczej:

```text
czy split poprawia stratę na tyle mocno,
żeby opłacało się dodać kolejne liście po uwzględnieniu kar regularyzacyjnych?
```

Dlatego w formule gain pojawiają się parametry:

- `lambda` / `reg_lambda` — kara za zbyt duże wartości liści,
- `gamma` — minimalny zysk wymagany, żeby w ogóle zaakceptować split.

Jeżeli `gain <= 0`, split nie jest wart dodania. To jest jeden z powodów, dla których XGBoost zwykle lepiej kontroluje przeuczenie niż naiwnie budowany boosting.


In [ ]:
# Ćwiczenie B: półręczny krok XGBoost dla tego samego splitu.
#
# Uwaga na znak:
# - wcześniej residual liczyliśmy jako y - p,
# - w XGBoost gradient log-loss zapisujemy zwykle jako g = p - y.
#
# Hessian dla log-loss:
#     h = p * (1 - p)
#
# Wartość liścia:
#     w = -sum(g) / (sum(h) + lambda)
#
# Gain splitu:
#     gain = 0.5 * [
#         G_left^2 / (H_left + lambda)
#       + G_right^2 / (H_right + lambda)
#       - G_parent^2 / (H_parent + lambda)
#     ] - gamma

mini_xgb = mini.copy()
mini_xgb["p0"] = p0_ref
mini_xgb["g"] = mini_xgb["p0"] - mini_xgb["odejdzie_num"]
mini_xgb["h"] = mini_xgb["p0"] * (1 - mini_xgb["p0"])

lambda_reg = 1.0
gamma = 0.0

left = mini_xgb.loc[~mask_duzo_reklamacji]
right = mini_xgb.loc[mask_duzo_reklamacji]
parent = mini_xgb

# TODO B1: policz sumy G i H dla obu gałęzi.
#
# G = suma gradientów g
# H = suma Hessianów h

G_left_recznie = ...
H_left_recznie = ...
G_right_recznie = ...
H_right_recznie = ...

G_left_ref = left["g"].sum()
H_left_ref = left["h"].sum()
G_right_ref = right["g"].sum()
H_right_ref = right["h"].sum()
G_parent_ref = parent["g"].sum()
H_parent_ref = parent["h"].sum()

check_close("G_left", G_left_recznie, G_left_ref)
check_close("H_left", H_left_recznie, H_left_ref)
check_close("G_right", G_right_recznie, G_right_ref)
check_close("H_right", H_right_recznie, H_right_ref)

# TODO B2: zaimplementuj dwie funkcje.

def xgb_leaf_weight_student(G, H, lambda_reg):
    # Pseudokod:
    # 1. w liczniku weź minus G,
    # 2. w mianowniku weź H + lambda_reg,
    # 3. zwróć iloraz.
    return ...


def xgb_split_gain_student(G_left, H_left, G_right, H_right, G_parent, H_parent, lambda_reg, gamma):
    # Pseudokod:
    # 1. score_left = G_left**2 / (H_left + lambda_reg)
    # 2. score_right = G_right**2 / (H_right + lambda_reg)
    # 3. score_parent = G_parent**2 / (H_parent + lambda_reg)
    # 4. gain = 0.5 * (score_left + score_right - score_parent) - gamma
    # 5. zwróć gain.
    return ...

w_left = xgb_leaf_weight_student(G_left_ref, H_left_ref, lambda_reg)
w_right = xgb_leaf_weight_student(G_right_ref, H_right_ref, lambda_reg)
gain = xgb_split_gain_student(
    G_left_ref, H_left_ref,
    G_right_ref, H_right_ref,
    G_parent_ref, H_parent_ref,
    lambda_reg, gamma,
)

if w_left is Ellipsis or w_right is Ellipsis or gain is Ellipsis:
    print("Uzupełnij funkcje xgb_leaf_weight_student i xgb_split_gain_student.")
else:
    print("waga liścia: reklamacje < 2  =", round(w_left, 6))
    print("waga liścia: reklamacje >= 2 =", round(w_right, 6))
    print("gain splitu =", round(gain, 6))

# Tabela pomocnicza: gradienty i Hessiany dla obserwacji.
display(mini_xgb[["klient", "odejdzie_num", "p0", "g", "h", "liczba_reklamacji"]].round(3))


## 4. Dane churn i preprocessing

Po mini-przykładzie wracamy do większego zbioru churn. Dane są syntetyczne, ale mają kontrolowaną logikę:

- więcej reklamacji zwiększa ryzyko odejścia,
- krótka umowa i umowa miesięczna zwiększają ryzyko odejścia,
- dłuższy staż, większa aktywność i rabat zmniejszają ryzyko odejścia.

Tak jak wcześniej, używamy `Pipeline` i `ColumnTransformer`, bo mamy cechy liczbowe, kategoryczne i braki danych.


In [ ]:
def make_churn_data(n=320, random_state=42):
    rng = np.random.default_rng(random_state)

    miesiace = rng.integers(1, 60, n)
    reklamacje = rng.poisson(1.1, n)
    logowania = np.round(np.clip(rng.normal(4.8, 1.9, n), 0, None), 1)
    plan = rng.choice(["basic", "standard", "premium"], n, p=[0.48, 0.34, 0.18])
    umowa = rng.choice(["miesieczna", "roczna", "dwuletnia"], n, p=[0.47, 0.35, 0.18])
    rabat = rng.choice(["tak", "nie"], n, p=[0.33, 0.67])

    logit_true = (
        1.2
        - 0.045 * miesiace
        + 0.55 * reklamacje
        - 0.32 * logowania
        + 0.75 * (umowa == "miesieczna")
        - 0.65 * (umowa == "dwuletnia")
        + 0.35 * (plan == "basic")
        - 0.30 * (plan == "premium")
        - 0.45 * (rabat == "tak")
        + rng.normal(0, 0.55, n)
    )
    p = sigmoid(logit_true)
    odejdzie = rng.binomial(1, p)

    df = pd.DataFrame({
        "klient_id": range(1, n + 1),
        "miesiace_umowy": miesiace,
        "liczba_reklamacji": reklamacje,
        "srednie_logowania_tyg": logowania,
        "plan": plan,
        "umowa": umowa,
        "rabat": rabat,
        "odejdzie": np.where(odejdzie == 1, "tak", "nie"),
    })

    missing_logins = rng.choice(df.index, size=max(4, n // 25), replace=False)
    missing_plan = rng.choice(df.index, size=max(3, n // 35), replace=False)
    df.loc[missing_logins, "srednie_logowania_tyg"] = np.nan
    df.loc[missing_plan, "plan"] = np.nan

    return df


churn = make_churn_data()

num_cols = ["miesiace_umowy", "liczba_reklamacji", "srednie_logowania_tyg"]
cat_cols = ["plan", "umowa", "rabat"]
target = "odejdzie"
positive_label = "tak"

X = churn.drop(columns=[target, "klient_id"])
y_text = churn[target]

label_encoder = LabelEncoder()
y_num = label_encoder.fit_transform(y_text)
positive_code = int(label_encoder.transform([positive_label])[0])

X_train, X_test, y_train_text, y_test_text, y_train_num, y_test_num = train_test_split(
    X,
    y_text,
    y_num,
    test_size=0.30,
    random_state=42,
    stratify=y_num,
)

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_one_hot_encoder()),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

print("Rozmiar danych:", churn.shape)
print("Mapowanie etykiet:")
for label, value in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"  {label} -> {value}")

display(churn.head())
display(churn["odejdzie"].value_counts(normalize=True).rename("proporcja"))


## 5. Modele porównawcze

Porównamy tę samą rodzinę metod, żeby zobaczyć, co zmienia się po drodze:

1. **Baseline** — zawsze przewiduje klasę większościową,
2. **Decision Tree** — jedno drzewo,
3. **Random Forest** — wiele niezależnych drzew,
4. **Gradient Boosting** — wiele małych drzew dodawanych sekwencyjnie,
5. **XGBoost** — jeżeli pakiet jest dostępny.

W churn sama `accuracy` może być myląca, więc patrzymy też na:

- `recall_tak` — ile faktycznych odejść wykryliśmy,
- `precision_tak` — ile alarmów było trafionych,
- `f1_tak` — kompromis precision/recall,
- `roc_auc` — jakość rankingu ryzyka.


In [ ]:
def positive_proba(estimator, X_values):
    classes = list(estimator.classes_)
    return estimator.predict_proba(X_values)[:, classes.index(positive_code)]


def evaluate_numeric_model(name, estimator, X_values, y_true_num):
    pred = estimator.predict(X_values)
    if hasattr(estimator, "predict_proba"):
        proba = positive_proba(estimator, X_values)
        roc_auc = roc_auc_score(y_true_num, proba)
        loss = log_loss(y_true_num, proba, labels=[0, 1])
    else:
        roc_auc = np.nan
        loss = np.nan

    return {
        "model": name,
        "accuracy": accuracy_score(y_true_num, pred),
        "precision_tak": precision_score(y_true_num, pred, pos_label=positive_code, zero_division=0),
        "recall_tak": recall_score(y_true_num, pred, pos_label=positive_code, zero_division=0),
        "f1_tak": f1_score(y_true_num, pred, pos_label=positive_code, zero_division=0),
        "roc_auc": roc_auc,
        "log_loss": loss,
    }


baseline = DummyClassifier(strategy="most_frequent")

tree_pipe = Pipeline([
    ("prep", preprocess),
    ("model", DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=8,
        random_state=42,
    )),
])

forest_pipe = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=7,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=1,
    )),
])

gb_pipe = Pipeline([
    ("prep", preprocess),
    ("model", GradientBoostingClassifier(
        n_estimators=120,
        learning_rate=0.05,
        max_depth=2,
        subsample=0.90,
        random_state=42,
    )),
])

baseline.fit(X_train, y_train_num)
tree_pipe.fit(X_train, y_train_num)
forest_pipe.fit(X_train, y_train_num)
gb_pipe.fit(X_train, y_train_num)

models = {
    "Baseline": baseline,
    "Decision Tree": tree_pipe,
    "Random Forest": forest_pipe,
    "Gradient Boosting": gb_pipe,
}

if XGB_AVAILABLE:
    n_positive = int((y_train_num == positive_code).sum())
    n_negative = int(len(y_train_num) - n_positive)
    scale_pos_weight = n_negative / n_positive

    xgb_pipe = Pipeline([
        ("prep", preprocess),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=160,
            learning_rate=0.04,
            max_depth=2,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=1,
            verbosity=0,
        )),
    ])
    xgb_pipe.fit(X_train, y_train_num)
    models["XGBoost"] = xgb_pipe
else:
    xgb_pipe = None
    print("XGBoost nie jest dostępny w tym środowisku:", XGB_IMPORT_ERROR)

comparison = pd.DataFrame([
    evaluate_numeric_model(name, model, X_test, y_test_num)
    for name, model in models.items()
]).sort_values("f1_tak", ascending=False)

display(comparison.round(3))


### Ćwiczenie C: interpretacja tabeli modeli

TODO: odpowiedz po uruchomieniu tabeli.

1. Który model ma najlepsze `f1_tak`?
2. Który model ma najlepsze `recall_tak`?
3. Czy model z najlepszą `accuracy` musi być najlepszy biznesowo?
4. Czy boosting zawsze musi wygrać z Random Forest?
5. Jeżeli model ma wysoki `roc_auc`, ale niski `recall_tak` przy progu 0.5, co można jeszcze zmienić?


## 6. Jak Gradient Boosting poprawia się po kolejnych drzewach?

`GradientBoostingClassifier` pozwala podejrzeć model po kolejnych etapach przez `staged_predict` i `staged_predict_proba`.

To jest bardzo ważne dydaktycznie: w Random Forest dokładamy kolejne niezależne drzewa do głosowania, a w Gradient Boostingu kolejne drzewa zmieniają aktualny model krok po kroku.

Patrzymy na:

```text
liczba dodanych drzew -> jakość na train i test
```

Jeżeli wynik treningowy poprawia się długo, ale testowy przestaje się poprawiać albo spada, zaczynamy podejrzewać przeuczenie.


In [ ]:
X_train_t = gb_pipe.named_steps["prep"].transform(X_train)
X_test_t = gb_pipe.named_steps["prep"].transform(X_test)
gb_model = gb_pipe.named_steps["model"]

gb_positive_idx = list(gb_model.classes_).index(positive_code)

stage_rows = []
for i, (pred_train, pred_test, proba_train_all, proba_test_all) in enumerate(
    zip(
        gb_model.staged_predict(X_train_t),
        gb_model.staged_predict(X_test_t),
        gb_model.staged_predict_proba(X_train_t),
        gb_model.staged_predict_proba(X_test_t),
    ),
    start=1,
):
    proba_train = proba_train_all[:, gb_positive_idx]
    proba_test = proba_test_all[:, gb_positive_idx]
    stage_rows.append({
        "n_drzewek": i,
        "log_loss_train": log_loss(y_train_num, proba_train, labels=[0, 1]),
        "log_loss_test": log_loss(y_test_num, proba_test, labels=[0, 1]),
        "accuracy_test": accuracy_score(y_test_num, pred_test),
        "f1_tak_test": f1_score(y_test_num, pred_test, pos_label=positive_code, zero_division=0),
        "roc_auc_test": roc_auc_score(y_test_num, proba_test),
    })

stages = pd.DataFrame(stage_rows)

rows_to_show = sorted(set([0, 1, 4, 9, 19, 49, len(stages) - 1]))
display(stages.iloc[rows_to_show].round(3))

plt.figure(figsize=(8, 4))
plt.plot(stages["n_drzewek"], stages["log_loss_train"], label="log_loss train")
plt.plot(stages["n_drzewek"], stages["log_loss_test"], label="log_loss test")
plt.xlabel("liczba dodanych drzewek")
plt.ylabel("log_loss")
plt.title("Gradient Boosting: strata po kolejnych drzewach")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(stages["n_drzewek"], stages["f1_tak_test"], label="f1 tak test")
plt.plot(stages["n_drzewek"], stages["roc_auc_test"], label="roc_auc test")
plt.xlabel("liczba dodanych drzewek")
plt.ylabel("metryka")
plt.title("Gradient Boosting: jakość testowa po kolejnych drzewach")
plt.legend()
plt.grid(True)
plt.show()


### Ćwiczenie D: odczyt krzywych staged prediction

TODO: odpowiedz po wykresach.

1. Czy `log_loss_train` zwykle maleje wraz z liczbą drzew?
2. Czy `log_loss_test` musi maleć cały czas?
3. Po co obserwujemy wynik po kolejnych etapach?
4. Kiedy większe `n_estimators` może zaszkodzić?


## 7. `learning_rate` i `n_estimators`

To jest jeden z najważniejszych kompromisów w boostingu:

```text
większy learning_rate  -> większa poprawka po każdym drzewie, ale większe ryzyko przesadzenia
mniejszy learning_rate -> ostrożniejsze poprawki, zwykle potrzeba więcej drzew
```

Dlatego często stroi się te parametry razem, a nie osobno.


In [ ]:
settings = [
    {"learning_rate": 0.20, "n_estimators": 40},
    {"learning_rate": 0.10, "n_estimators": 80},
    {"learning_rate": 0.05, "n_estimators": 160},
    {"learning_rate": 0.02, "n_estimators": 260},
]

lr_rows = []
for params in settings:
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", GradientBoostingClassifier(
            learning_rate=params["learning_rate"],
            n_estimators=params["n_estimators"],
            max_depth=2,
            subsample=0.90,
            random_state=42,
        )),
    ])
    pipe.fit(X_train, y_train_num)
    row = evaluate_numeric_model("Gradient Boosting", pipe, X_test, y_test_num)
    lr_rows.append({
        **params,
        "learning_rate_x_n_estimators": params["learning_rate"] * params["n_estimators"],
        **row,
    })

lr_table = pd.DataFrame(lr_rows).drop(columns=["model"])
display(lr_table.round(3))


### Ćwiczenie E: interpretacja `learning_rate`

TODO: odpowiedz po tabeli.

1. Czy największy `learning_rate` daje zawsze najlepszy wynik?
2. Czy najmniejszy `learning_rate` daje zawsze najlepszy wynik?
3. Dlaczego nie porównujemy `learning_rate` bez patrzenia na `n_estimators`?
4. Co byś zwiększył, jeżeli `learning_rate` jest bardzo mały i model niedoucza się?


## 8. XGBoost w praktyce: które parametry są nowymi „dźwigniami”?

W XGBoost dochodzą parametry, które kontrolują złożoność i losowość:

| Parametr | Intuicja |
|---|---|
| `max_depth` | jak złożone może być pojedyncze drzewo-poprawka |
| `learning_rate` | jak mocno dokładamy każde drzewo |
| `n_estimators` | ile drzew-poprawek dodajemy |
| `subsample` | jaki procent obserwacji bierze jedno drzewo |
| `colsample_bytree` | jaki procent cech bierze jedno drzewo |
| `reg_lambda` | kara za zbyt duże wartości liści |
| `gamma` | minimalny zysk wymagany do wykonania splitu |
| `scale_pos_weight` | pomoc przy niezbalansowanej klasie pozytywnej |

To nie są przypadkowe parametry — one odpowiadają za kontrolowanie tego, jak agresywnie boosting poprawia błędy.


In [ ]:
if not XGB_AVAILABLE:
    print("Pomijamy eksperyment XGBoost, bo pakiet xgboost nie jest dostępny.")
else:
    xgb_settings = [
        {
            "wariant": "płytkie drzewa, umiarkowany krok",
            "max_depth": 2,
            "learning_rate": 0.05,
            "n_estimators": 140,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_lambda": 1.0,
        },
        {
            "wariant": "głębsze drzewa",
            "max_depth": 4,
            "learning_rate": 0.05,
            "n_estimators": 140,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_lambda": 1.0,
        },
        {
            "wariant": "mniejszy krok, więcej drzew",
            "max_depth": 2,
            "learning_rate": 0.025,
            "n_estimators": 260,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_lambda": 1.0,
        },
        {
            "wariant": "mocniejsza regularizacja",
            "max_depth": 2,
            "learning_rate": 0.05,
            "n_estimators": 140,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "reg_lambda": 5.0,
        },
    ]

    xgb_rows = []
    for params in xgb_settings:
        pipe = Pipeline([
            ("prep", preprocess),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                max_depth=params["max_depth"],
                learning_rate=params["learning_rate"],
                n_estimators=params["n_estimators"],
                subsample=params["subsample"],
                colsample_bytree=params["colsample_bytree"],
                reg_lambda=params["reg_lambda"],
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                n_jobs=1,
                verbosity=0,
            )),
        ])
        pipe.fit(X_train, y_train_num)
        metrics = evaluate_numeric_model("XGBoost", pipe, X_test, y_test_num)
        xgb_rows.append({**params, **metrics})

    xgb_param_table = pd.DataFrame(xgb_rows).drop(columns=["model"])
    display(xgb_param_table.round(3))


### Ćwiczenie F: interpretacja parametrów XGBoost

TODO: odpowiedz po tabeli.

1. Co może się stać, gdy zwiększamy `max_depth`?
2. Po co zmniejszać `subsample` albo `colsample_bytree`?
3. Co intuicyjnie robi większe `reg_lambda`?
4. Czy wariant z najlepszym `accuracy` musi być najlepszy dla churn?


## 9. Ważność cech i ranking ryzyka dla modelu wybranego według `f1_tak`

Boosting zwraca prawdopodobieństwo odejścia. Dzięki temu możemy:

1. zbudować ranking klientów według ryzyka,
2. dobrać próg decyzyjny,
3. sprawdzić, które cechy były najważniejsze dla modelu.

`feature_importances_` w modelach drzewiastych oznacza zwykle:

```text
znormalizowany wkład cechy w poprawę splitów w drzewach modelu
```

Dla klasycznych modeli scikit-learn, takich jak `RandomForestClassifier` i `GradientBoostingClassifier`, jest to najczęściej suma spadków nieczystości przypisana do splitów używających danej cechy. Dla XGBoost podobna intuicja dotyczy miary typu `gain`: cecha dostaje większą wartość, jeżeli splity z jej udziałem mocno poprawiały funkcję celu.

Czyli praktycznie:

```text
wysoka ważność = model często i skutecznie używał tej cechy do poprawiania predykcji
niska ważność  = cecha rzadko pomagała albo dawała małą poprawę
```

To opis działania wytrenowanego modelu predykcyjnego. Notatka o przyczynowości jest dodatkiem: sama ważność cechy nie wystarcza, żeby powiedzieć, że cecha powoduje churn.


In [ ]:
# Wybieramy model według najwyższego f1_tak z tabeli comparison.
# Dzięki temu dalsze sekcje rzeczywiście analizują najlepszy model według przyjętej metryki,
# a nie automatycznie XGBoost tylko dlatego, że pakiet jest dostępny.
best_model_name = comparison.iloc[0]["model"]
best_model = models[best_model_name]
print("Model wybrany według najwyższego f1_tak:", best_model_name)

if isinstance(best_model, Pipeline):
    feature_names = best_model.named_steps["prep"].get_feature_names_out()
    model_inside = best_model.named_steps["model"]

    if hasattr(model_inside, "feature_importances_"):
        importance = pd.DataFrame({
            "cecha_po_transformacji": feature_names,
            "importance": model_inside.feature_importances_,
        }).sort_values("importance", ascending=False)
        importance["udział_%"] = 100 * importance["importance"]

        print("Suma importance:", round(float(importance["importance"].sum()), 6))
        display(importance.head(10).round({"importance": 4, "udział_%": 2}))

        top = importance.head(10).iloc[::-1]
        plt.figure(figsize=(8, 4))
        plt.barh(top["cecha_po_transformacji"], top["importance"])
        plt.xlabel("feature_importance / wkład cechy w splity")
        plt.title(f"Najważniejsze cechy: {best_model_name}")
        plt.show()
    else:
        print("Wybrany model nie udostępnia feature_importances_.")
else:
    print("Wybrany model nie jest pipeline'em drzewiastym, więc pomijamy feature_importances_.")

risk = X_test.copy()
risk["prawdziwa_etykieta"] = label_encoder.inverse_transform(y_test_num)
risk["p_odejdzie_tak"] = positive_proba(best_model, X_test)
risk["predykcja_przy_progu_0_5"] = label_encoder.inverse_transform(best_model.predict(X_test))

display(risk.sort_values("p_odejdzie_tak", ascending=False).head(10))


### Dodatkowa kontrola: permutation importance

Ważność z `feature_importances_` mówi, jak model używał cech wewnątrz drzew.

`permutation_importance` mierzy coś bardziej zewnętrznego:

```text
mieszamy jedną cechę w zbiorze testowym i sprawdzamy,
o ile spada f1_tak.
```

Jeżeli po wymieszaniu cechy jakość mocno spada, to model potrzebował tej informacji do dobrej predykcji.


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

f1_numeric_scorer = make_scorer(f1_score, pos_label=positive_code, zero_division=0)

perm = permutation_importance(
    best_model,
    X_test,
    y_test_num,
    scoring=f1_numeric_scorer,
    n_repeats=10,
    random_state=42,
    n_jobs=1,
)

permutation_table = pd.DataFrame({
    "cecha_oryginalna": X_test.columns,
    "spadek_f1_tak_po_permutacji": perm.importances_mean,
    "odchylenie": perm.importances_std,
}).sort_values("spadek_f1_tak_po_permutacji", ascending=False)

display(permutation_table.round(4))


## 10. Próg decyzyjny

Model zwraca prawdopodobieństwo klasy `tak`. Klasa `tak/nie` powstaje dopiero po wybraniu progu.

```text
p_odejdzie_tak >= próg  -> przewidujemy "tak"
p_odejdzie_tak <  próg  -> przewidujemy "nie"
```

Tak samo jak w Random Forest, próg `0.50` jest tylko punktem startowym. Różne progi dają różny kompromis między `precision_tak`, `recall_tak` i `f1_tak`.

W tej sekcji pokazujemy także `TP`, `FP`, `FN`, `TN`, żeby próg dało się uzasadnić konkretnymi liczbami, a nie tylko jedną metryką.


In [ ]:
from sklearn.metrics import confusion_matrix

proba_tak = positive_proba(best_model, X_test)
negative_code = int([code for code in np.unique(y_test_num) if code != positive_code][0])


def threshold_metrics_table_numeric(y_true_num, proba_positive, thresholds):
    """Tabela metryk dla różnych progów klasy 'tak' przy etykietach zakodowanych liczbowo."""
    rows = []
    for threshold in thresholds:
        pred_thr = np.where(proba_positive >= threshold, positive_code, negative_code)

        # labels=[positive_code, negative_code] daje układ:
        # [[TP, FN],
        #  [FP, TN]]
        cm = confusion_matrix(y_true_num, pred_thr, labels=[positive_code, negative_code])
        tp = int(cm[0, 0])
        fn = int(cm[0, 1])
        fp = int(cm[1, 0])
        tn = int(cm[1, 1])

        rows.append({
            "threshold": float(threshold),
            "pred_tak": int(tp + fp),
            "TP_zlapany_churn": tp,
            "FP_falszywy_alarm": fp,
            "FN_przeoczony_churn": fn,
            "TN_poprawne_nie": tn,
            "precision_tak": precision_score(y_true_num, pred_thr, pos_label=positive_code, zero_division=0),
            "recall_tak": recall_score(y_true_num, pred_thr, pos_label=positive_code, zero_division=0),
            "f1_tak": f1_score(y_true_num, pred_thr, pos_label=positive_code, zero_division=0),
        })
    return pd.DataFrame(rows)


thresholds = threshold_metrics_table_numeric(
    y_test_num,
    proba_tak,
    thresholds=np.linspace(0.20, 0.80, 13),
)

display(thresholds.round(3))

plt.figure(figsize=(8, 4))
plt.plot(thresholds["threshold"], thresholds["precision_tak"], marker="o", label="precision_tak")
plt.plot(thresholds["threshold"], thresholds["recall_tak"], marker="o", label="recall_tak")
plt.plot(thresholds["threshold"], thresholds["f1_tak"], marker="o", label="f1_tak")
plt.xlabel("próg dla klasy tak")
plt.ylabel("wartość metryki")
plt.title(f"Wpływ progu decyzyjnego: {best_model_name}")
plt.legend()
plt.grid(True)
plt.show()

best_f1_threshold = thresholds.sort_values(["f1_tak", "recall_tak"], ascending=False).iloc[0]
print("Próg z najwyższym f1_tak w tej siatce:")
display(best_f1_threshold.to_frame().T.round(3))


### Minićwiczenie G: wybierz próg dla najlepszego modelu

Na podstawie tabeli wybierz próg, który pasuje do celu:

```text
Chcemy dobrego f1_tak, ale nie chcemy bardzo niskiego recall_tak.
```

Możesz zacząć od progu z najwyższym `f1_tak`, a potem sprawdzić, czy liczba `FN_przeoczony_churn` nie jest zbyt duża.


In [ ]:
# TODO G: wybierz próg dla najlepszego modelu i krótko go uzasadnij.
#
# Wskazówka:
# - best_f1_threshold pokazuje próg z najwyższym f1_tak w sprawdzonej siatce.
# - możesz wybrać ten próg albo inny, jeśli lepiej pasuje do celu biznesowego.

wybrany_prog_boosting = ...
uzasadnienie_progu = "..."

if not all_filled(wybrany_prog_boosting) or uzasadnienie_progu.strip() in {"", "..."}:
    print("Uzupełnij wybrany_prog_boosting oraz uzasadnienie_progu.")
else:
    wybrany_wiersz = thresholds[np.isclose(thresholds["threshold"], wybrany_prog_boosting)]
    if wybrany_wiersz.empty:
        print("Ten próg nie występuje w tabeli thresholds.")
    else:
        print("Wybrany próg:", wybrany_prog_boosting)
        print("Uzasadnienie:", uzasadnienie_progu)
        display(wybrany_wiersz.round(3))


## 11. Macierz pomyłek dla najlepszego modelu z notebooka

Na końcu sprawdzamy raport klasyfikacji dla domyślnej decyzji modelu, czyli w praktyce dla progu około `0.50` w klasyfikacji binarnej.

Jeżeli w poprzedniej sekcji wybraliśmy inny próg, to metryki z tabeli progów są ważniejsze dla tej konkretnej decyzji biznesowej.


In [ ]:
pred_best = best_model.predict(X_test)

print(best_model_name)
print(classification_report(
    y_test_num,
    pred_best,
    target_names=label_encoder.classes_,
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test_num,
    pred_best,
    display_labels=label_encoder.classes_,
)
plt.title(f"Macierz pomyłek: {best_model_name}")
plt.show()


## Pytania końcowe

TODO: odpowiedz własnymi słowami.

1. Czym różni się Random Forest od Gradient Boostingu?
2. Co oznacza, że boosting buduje drzewa sekwencyjnie?
3. Co robi `learning_rate`?
4. Dlaczego w klasyfikacji binarnej mówimy o score/logit, a dopiero potem o prawdopodobieństwie?
5. Czym różni się uproszczony residual `y - p` od gradientu używanego w XGBoost?
6. Co dodaje XGBoost względem zwykłego Gradient Boostingu?
7. Dlaczego w churn warto analizować próg decyzyjny?
8. Co oznacza `feature_importances_` w modelach boostingowych i czym różni się od permutation importance?


## Końcowa mapa całego pakietu 4.1–4.4

Najważniejsza sekwencja pojęć wygląda tak:

```text
4.1 Jedno drzewo klasyfikacyjne
    cechy binarne -> Gini -> najlepszy split -> klasa

4.2A Jedno drzewo klasyfikacyjne z cechami liczbowymi
    cechy liczbowe -> szukanie progów -> Gini -> klasa

4.2B Jedno drzewo regresyjne
    cechy liczbowe -> szukanie progów -> MSE/MAE -> liczba

4.3 Random Forest
    wiele niezależnych drzew -> bootstrap + losowanie cech -> głosowanie/średnia

4.4 Gradient Boosting / XGBoost
    wiele małych drzew sekwencyjnie -> każde drzewo dodaje poprawkę -> learning_rate i regularizacja
```

Trzy zdania kontrolne dla studenta:

1. **Progowanie nie oznacza regresji** — progi występują też w klasyfikacji.
2. **Random Forest nie zmienia pojedynczego splitu** — zmienia sposób łączenia wielu drzew.
3. **Boosting nie jest lasem głosujących drzew** — to sekwencja poprawek dodawanych do aktualnego modelu.